In [2]:
import os

os.makedirs(
    "Task2-RAG-From-Scratch/data/documents",
    exist_ok=True
)

print("Task 2 folders created!")

Task 2 folders created!


In [3]:

import requests

pdf_url = (
    "https://nvlpubs.nist.gov/nistpubs/"
    "ai/NIST.AI.100-1.pdf"
)

pdf_path = (
    "Task2-RAG-From-Scratch/data/documents/"
    "nist_ai_rmf_1_0.pdf"
)

response = requests.get(pdf_url, timeout=60)
response.raise_for_status()

with open(pdf_path, "wb") as f:
    f.write(response.content)

print("PDF downloaded successfully!")
print("File:", pdf_path)
print("Size:", round(len(response.content) / (1024 * 1024), 2), "MB")


PDF downloaded successfully!
File: Task2-RAG-From-Scratch/data/documents/nist_ai_rmf_1_0.pdf
Size: 1.86 MB


In [4]:
!pip install -q pypdf

In [5]:

from pypdf import PdfReader

reader = PdfReader(pdf_path)

total_pages = len(reader.pages)

print("Document:", "NIST AI RMF 1.0")
print("Total PDF pages:", total_pages)

assert total_pages >= 30, (
    "Document must contain at least 30 pages!"
)

print("\nDocument validation passed!")


Document: NIST AI RMF 1.0
Total PDF pages: 48

Document validation passed!


In [6]:
# Extract text page by page

from pypdf import PdfReader
import os
import json

# PDF location
pdf_path = (
    "Task2-RAG-From-Scratch/data/documents/"
    "nist_ai_rmf_1_0.pdf"
)

# Read PDF
reader = PdfReader(pdf_path)

# Store extracted pages
extracted_pages = []

for page_number, page in enumerate(reader.pages, start=1):

    # Extract text
    text = page.extract_text() or ""

    # Clean unnecessary whitespace
    text = " ".join(text.split())

    extracted_pages.append({
        "page_number": page_number,
        "text": text
    })

print("PDF text extraction completed!")
print("Total pages:", len(extracted_pages))
print(
    "Pages containing text:",
    sum(bool(page["text"]) for page in extracted_pages)
)
print(
    "Total extracted characters:",
    sum(len(page["text"]) for page in extracted_pages)
)


PDF text extraction completed!
Total pages: 48
Pages containing text: 48
Total extracted characters: 106427


In [7]:

# Display first three pages
for page in extracted_pages[:3]:

    print("\n" + "=" * 60)
    print("PAGE:", page["page_number"])
    print("=" * 60)

    print(page["text"][:1000])



PAGE: 1
NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0)

PAGE: 2
NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0) This publication is available free of charge from: https://doi.org/10.6028/NIST.AI.100-1 January 2023 U.S. Department of Commerce Gina M. Raimondo, Secretary National Institute of Standards and Technology Laurie E. Locascio, NIST Director and Under Secretary of Commerce for Standards and Technology

PAGE: 3
Certain commercial entities, equipment, or materials may be identified in this document in order to describe an experimental procedure or concept adequately. Such identification is not intended to imply recommenda- tion or endorsement by the National Institute of Standards and Technology, nor is it intended to imply that the entities, materials, or equipment are necessarily the best available for the purpose. This publication is available free of charge from: https://doi.org/10.6028/NIST.AI.100-1 Update Schedule a

In [8]:
# Check for empty pages

empty_pages = [
    page["page_number"]
    for page in extracted_pages
    if not page["text"]
]

print("Empty pages:", empty_pages)
print("Number of empty pages:", len(empty_pages))

if len(empty_pages) == 0:
    print("All pages contain extractable text.")
else:
    print("Some pages need inspection.")


Empty pages: []
Number of empty pages: 0
All pages contain extractable text.


In [9]:
# Save the extracted text
# We will save two files:
# - extracted_pages.json – preserves each page number and its text.
# - extracted_text.txt – a readable text version of the document.


output_dir = "Task2-RAG-From-Scratch/data"

os.makedirs(output_dir, exist_ok=True)

# Save page-wise JSON
json_path = os.path.join(
    output_dir,
    "extracted_pages.json"
)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        extracted_pages,
        f,
        ensure_ascii=False,
        indent=4
    )

# Save readable text file
text_path = os.path.join(
    output_dir,
    "extracted_text.txt"
)

with open(text_path, "w", encoding="utf-8") as f:

    for page in extracted_pages:

        f.write(
            f"\n\n--- PAGE {page['page_number']} ---\n\n"
        )

        f.write(page["text"])

print("Extraction files saved successfully!")
print(json_path)
print(text_path)


Extraction files saved successfully!
Task2-RAG-From-Scratch/data/extracted_pages.json
Task2-RAG-From-Scratch/data/extracted_text.txt


In [10]:
# Create chunks of two different sizes
# We’ll use:
# - Small chunks: 200 words, with 40 words overlap.
# - Large chunks: 400 words, with 40 words overlap.


import json
import os

# Load extracted pages
json_path = "Task2-RAG-From-Scratch/data/extracted_pages.json"

with open(json_path, "r", encoding="utf-8") as f:
    pages = json.load(f)

def create_chunks(pages, chunk_size, overlap):
    chunks = []

    for page in pages:
        words = page["text"].split()
        start = 0

        while start < len(words):
            end = start + chunk_size
            chunk_words = words[start:end]

            if not chunk_words:
                break

            chunks.append({
                "chunk_id": len(chunks),
                "text": " ".join(chunk_words),
                "page_number": page["page_number"],
                "word_count": len(chunk_words)
            })

            if end >= len(words):
                break

            start = end - overlap

    return chunks

# Create both chunk versions
small_chunks = create_chunks(pages, chunk_size=200, overlap=40)
large_chunks = create_chunks(pages, chunk_size=400, overlap=40)

# Save chunk files
output_dir = "Task2-RAG-From-Scratch/data"
os.makedirs(output_dir, exist_ok=True)

with open(os.path.join(output_dir, "chunks_200.json"), "w", encoding="utf-8") as f:
    json.dump(small_chunks, f, indent=2, ensure_ascii=False)

with open(os.path.join(output_dir, "chunks_400.json"), "w", encoding="utf-8") as f:
    json.dump(large_chunks, f, indent=2, ensure_ascii=False)

# Display results
print("Small chunks (200 words):", len(small_chunks))
print("Large chunks (400 words):", len(large_chunks))

print("\nExample small chunk:")
print(small_chunks[0])

print("\nExample large chunk:")
print(large_chunks[0])


Small chunks (200 words): 114
Large chunks (400 words): 66

Example small chunk:
{'chunk_id': 0, 'text': 'NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0)', 'page_number': 1, 'word_count': 11}

Example large chunk:
{'chunk_id': 0, 'text': 'NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0)', 'page_number': 1, 'word_count': 11}


In [11]:
# Generate embeddings

!pip install -q sentence-transformers

import json
import numpy as np
import os
from sentence_transformers import SentenceTransformer

# Load both chunk files
data_dir = "Task2-RAG-From-Scratch/data"

with open(os.path.join(data_dir, "chunks_200.json"), "r", encoding="utf-8") as f:
    small_chunks = json.load(f)

with open(os.path.join(data_dir, "chunks_400.json"), "r", encoding="utf-8") as f:
    large_chunks = json.load(f)

# Load embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Extract text
small_texts = [chunk["text"] for chunk in small_chunks]
large_texts = [chunk["text"] for chunk in large_chunks]

# Generate embeddings
small_embeddings = model.encode(
    small_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

large_embeddings = model.encode(
    large_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Save embeddings
np.save(os.path.join(data_dir, "embeddings_200.npy"), small_embeddings)
np.save(os.path.join(data_dir, "embeddings_400.npy"), large_embeddings)

print("Small embedding shape:", small_embeddings.shape)
print("Large embedding shape:", large_embeddings.shape)
print("Embedding generation completed!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Small embedding shape: (114, 384)
Large embedding shape: (66, 384)
Embedding generation completed!


In [12]:
# Prepare 15 evaluation questions

import json

json_path = "Task2-RAG-From-Scratch/data/extracted_pages.json"

with open(json_path, "r", encoding="utf-8") as f:
    pages = json.load(f)

# Print page numbers and the first 1,200 characters
# from pages that contain useful text.
for page in pages:
    text = page["text"]

    if len(text) > 200:
        print("\n" + "=" * 80)
        print("PDF PAGE:", page["page_number"])
        print(text[:1200])



PDF PAGE: 2
NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0) This publication is available free of charge from: https://doi.org/10.6028/NIST.AI.100-1 January 2023 U.S. Department of Commerce Gina M. Raimondo, Secretary National Institute of Standards and Technology Laurie E. Locascio, NIST Director and Under Secretary of Commerce for Standards and Technology

PDF PAGE: 3
Certain commercial entities, equipment, or materials may be identified in this document in order to describe an experimental procedure or concept adequately. Such identification is not intended to imply recommenda- tion or endorsement by the National Institute of Standards and Technology, nor is it intended to imply that the entities, materials, or equipment are necessarily the best available for the purpose. This publication is available free of charge from: https://doi.org/10.6028/NIST.AI.100-1 Update Schedule and Versions The Artificial Intelligence Risk Management Framework (AI RMF) is 

In [13]:
# Create the evaluation dataset

import json
import os

data_dir = "Task2-RAG-From-Scratch/data"

# Manually prepared questions and supporting passages
evaluation_questions = [
    {
        "id": 1,
        "question": "What is the main purpose of the NIST AI Risk Management Framework?",
        "gold_page": 9,
        "gold_passage": "AI risk management offers a path to minimize potential negative impacts of AI systems, such as threats to civil liberties and rights, while also providing opportunities to maximize positive impacts."
    },
    {
        "id": 2,
        "question": "How does the AI RMF define risk?",
        "gold_page": 9,
        "gold_passage": "Risk refers to the composite measure of an event's probability of occurring and the magnitude or degree of the consequences of the corresponding event."
    },
    {
        "id": 3,
        "question": "What does risk tolerance mean in the context of AI risk management?",
        "gold_page": 12,
        "gold_passage": "Risk tolerance refers to the organization's or AI actor's readiness to bear the risk in order to achieve its objectives."
    },
    {
        "id": 4,
        "question": "What should organizations do when an AI system presents unacceptable negative risk levels?",
        "gold_page": 13,
        "gold_passage": "Development and deployment should cease in a safe manner until risks can be sufficiently managed."
    },
    {
        "id": 5,
        "question": "Why is organizational commitment important for effective AI risk management?",
        "gold_page": 14,
        "gold_passage": "Effective risk management is realized through organizational commitment at senior levels and may require cultural change within an organization or industry."
    },
    {
        "id": 6,
        "question": "What characteristics does NIST identify as necessary for trustworthy AI systems?",
        "gold_page": 17,
        "gold_passage": "Characteristics of trustworthy AI systems include: valid and reliable, safe, secure and resilient, accountable and transparent, explainable and interpretable, privacy-enhanced, and fair with harmful bias managed."
    },
    {
        "id": 7,
        "question": "How should AI system accuracy be measured according to the framework?",
        "gold_page": 19,
        "gold_passage": "Accuracy measurements should always be paired with clearly defined and realistic test sets that are representative of conditions of expected use and details about test methodology."
    },
    {
        "id": 8,
        "question": "What practical approaches can organizations use to improve AI safety?",
        "gold_page": 20,
        "gold_passage": "Other practical approaches for AI safety often relate to rigorous simulation and in-domain testing, real-time monitoring, and the ability to shut down, modify, or have human intervention into systems that deviate from intended or expected functionality."
    },
    {
        "id": 9,
        "question": "What is the difference between transparency, explainability, and interpretability?",
        "gold_page": 22,
        "gold_passage": "Transparency can answer the question of what happened in the system. Explainability can answer the question of how a decision was made in the system. Interpretability can answer the question of why a decision was made by the system and its meaning or context to the user."
    },
    {
        "id": 10,
        "question": "What are the three major categories of AI bias identified by NIST?",
        "gold_page": 23,
        "gold_passage": "NIST has identified three major categories of AI bias to be considered and managed: systemic, computational and statistical, and human-cognitive."
    },
    {
        "id": 11,
        "question": "What are the four functions of the AI RMF Core?",
        "gold_page": 25,
        "gold_passage": "The Core is composed of four functions: GOVERN, MAP, MEASURE, and MANAGE."
    },
    {
        "id": 12,
        "question": "Why is the GOVERN function considered cross-cutting?",
        "gold_page": 27,
        "gold_passage": "GOVERN is a cross-cutting function that is infused throughout AI risk management and enables the other functions of the process."
    },
    {
        "id": 13,
        "question": "What is the purpose of the MAP function?",
        "gold_page": 30,
        "gold_passage": "The MAP function is intended to enhance an organization's ability to identify risks and broader contributing factors."
    },
    {
        "id": 14,
        "question": "What does the MEASURE function involve?",
        "gold_page": 33,
        "gold_passage": "The MEASURE function employs quantitative, qualitative, or mixed-method tools, techniques, and methodologies to analyze, assess, benchmark, and monitor AI risk and related impacts."
    },
    {
        "id": 15,
        "question": "What are the main responsibilities of the MANAGE function?",
        "gold_page": 37,
        "gold_passage": "The MANAGE function entails allocating risk resources to respond to, recover from, and communicate about AI risks."
    }
]

# Save evaluation dataset
output_path = os.path.join(data_dir, "evaluation_questions.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_questions, f, indent=2, ensure_ascii=False)

print("Total evaluation questions:", len(evaluation_questions))
print("Saved successfully:", output_path)


Total evaluation questions: 15
Saved successfully: Task2-RAG-From-Scratch/data/evaluation_questions.json


In [14]:
# Implement top-5 retrieval and calculate Recall@5

import json
import numpy as np
import os

data_dir = "Task2-RAG-From-Scratch/data"

# Load evaluation questions
with open(os.path.join(data_dir, "evaluation_questions.json"), "r", encoding="utf-8") as f:
    questions = json.load(f)

# Load chunks and embeddings
with open(os.path.join(data_dir, "chunks_200.json"), "r", encoding="utf-8") as f:
    small_chunks = json.load(f)

with open(os.path.join(data_dir, "chunks_400.json"), "r", encoding="utf-8") as f:
    large_chunks = json.load(f)

small_embeddings = np.load(os.path.join(data_dir, "embeddings_200.npy"))
large_embeddings = np.load(os.path.join(data_dir, "embeddings_400.npy"))

# Load the same embedding model used earlier
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def retrieve_top_k(question, chunks, embeddings, k=5):
    # Convert question into an embedding
    query_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Cosine similarity using normalized vectors
    similarities = np.dot(embeddings, query_embedding)

    # Get indices of the top-k most similar chunks
    top_indices = np.argsort(similarities)[::-1][:k]

    results = []

    for index in top_indices:
        results.append({
            "chunk_id": chunks[index]["chunk_id"],
            "page_number": chunks[index]["page_number"],
            "similarity": float(similarities[index]),
            "text": chunks[index]["text"]
        })

    return results

def evaluate_recall_at_5(questions, chunks, embeddings):
    correct = 0
    detailed_results = []

    for item in questions:
        retrieved = retrieve_top_k(
            item["question"],
            chunks,
            embeddings,
            k=5
        )

        retrieved_pages = [
            result["page_number"] for result in retrieved
        ]

        # Success if the correct source page is in the top 5
        success = item["gold_page"] in retrieved_pages

        if success:
            correct += 1

        detailed_results.append({
            "question_id": item["id"],
            "question": item["question"],
            "gold_page": item["gold_page"],
            "retrieved_pages": retrieved_pages,
            "recall_at_5_hit": success,
            "top_results": retrieved
        })

    recall = correct / len(questions)

    return recall, detailed_results

# Evaluate both chunk sizes
small_recall, small_results = evaluate_recall_at_5(
    questions, small_chunks, small_embeddings
)

large_recall, large_results = evaluate_recall_at_5(
    questions, large_chunks, large_embeddings
)

print("=" * 55)
print("RETRIEVAL EVALUATION")
print("=" * 55)

print(f"200-word chunks Recall@5: {small_recall:.4f} ({small_recall*100:.2f}%)")
print(f"400-word chunks Recall@5: {large_recall:.4f} ({large_recall*100:.2f}%)")

# Save detailed results
with open(os.path.join(data_dir, "retrieval_results_200.json"), "w", encoding="utf-8") as f:
    json.dump(small_results, f, indent=2, ensure_ascii=False)

with open(os.path.join(data_dir, "retrieval_results_400.json"), "w", encoding="utf-8") as f:
    json.dump(large_results, f, indent=2, ensure_ascii=False)

print("\nDetailed retrieval results saved successfully!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RETRIEVAL EVALUATION
200-word chunks Recall@5: 0.9333 (93.33%)
400-word chunks Recall@5: 0.8000 (80.00%)

Detailed retrieval results saved successfully!


In [15]:
# Identify retrieval failures

import json
import os

data_dir = "Task2-RAG-From-Scratch/data"

with open(os.path.join(data_dir, "retrieval_results_200.json"), "r", encoding="utf-8") as f:
    small_results = json.load(f)

with open(os.path.join(data_dir, "retrieval_results_400.json"), "r", encoding="utf-8") as f:
    large_results = json.load(f)

def show_failures(results, chunk_label):
    failures = [item for item in results if not item["recall_at_5_hit"]]

    print("\n" + "=" * 70)
    print(f"FAILURES: {chunk_label}")
    print("Total failures:", len(failures))
    print("=" * 70)

    for item in failures:
        print("\nQuestion ID:", item["question_id"])
        print("Question:", item["question"])
        print("Correct page:", item["gold_page"])
        print("Retrieved pages:", item["retrieved_pages"])

        print("\nTop retrieved passages:")
        for rank, result in enumerate(item["top_results"], start=1):
            print(f"\nRank {rank}")
            print("Page:", result["page_number"])
            print("Similarity:", round(result["similarity"], 4))
            print("Text:", result["text"][:500])

show_failures(small_results, "200-word chunks")
show_failures(large_results, "400-word chunks")



FAILURES: 200-word chunks
Total failures: 1

Question ID: 1
Question: What is the main purpose of the NIST AI Risk Management Framework?
Correct page: 9
Retrieved pages: [1, 2, 6, 37, 8]

Top retrieved passages:

Rank 1
Page: 1
Similarity: 0.7687
Text: NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0)

Rank 2
Page: 2
Similarity: 0.7594
Text: NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0) This publication is available free of charge from: https://doi.org/10.6028/NIST.AI.100-1 January 2023 U.S. Department of Commerce Gina M. Raimondo, Secretary National Institute of Standards and Technology Laurie E. Locascio, NIST Director and Under Secretary of Commerce for Standards and Technology

Rank 3
Page: 6
Similarity: 0.7273
Text: NIST AI 100-1 AI RMF 1.0 Executive Summary Artificial intelligence (AI) technologies have significant potential to transform society and people’s lives – from commerce and health to transportation and cybersecu

In [16]:
# Build the final RAG question-answering system

!pip install -q transformers accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load Qwen instruction model
llm_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_name)

llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype="auto",
    device_map="auto"
)

print("LLM loaded successfully!")

def rag_answer(question):
    # Retrieve top 5 relevant chunks using the 200-word configuration
    retrieved = retrieve_top_k(
        question,
        small_chunks,
        small_embeddings,
        k=5
    )

    # Build document context with page references
    context = ""

    for i, result in enumerate(retrieved, start=1):
        context += (
            f"\n[Source {i} | PDF Page {result['page_number']}]\n"
            f"{result['text']}\n"
        )

    # Ground the response in retrieved content
    messages = [
        {
            "role": "system",
            "content": (
                "You are a document question-answering assistant. "
                "Answer using only the supplied context. "
                "Do not invent facts or use outside knowledge. "
                "If the context does not contain the answer, say "
                "'The retrieved document passages do not provide "
                "enough information to answer this question.' "
                "Keep the answer clear and concise. "
                "Mention the relevant PDF page number."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}\n\n"
                "Answer based only on the context above."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    source_pages = sorted(set(
        result["page_number"] for result in retrieved
    ))

    return {
        "question": question,
        "answer": answer,
        "source_pages": source_pages,
        "retrieved_passages": retrieved
    }

# Test the RAG system
test_question = "What are the four functions of the AI RMF Core?"

result = rag_answer(test_question)

print("\nQUESTION:")
print(result["question"])

print("\nGENERATED ANSWER:")
print(result["answer"])

print("\nRETRIEVED SOURCE PAGES:")
print(result["source_pages"])


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully!

QUESTION:
What are the four functions of the AI RMF Core?

GENERATED ANSWER:
The four functions of the AI RMF Core are:
GOVERN, MAP, MEASURE, and MANAGE.

RETRIEVED SOURCE PAGES:
[7, 8, 25, 26, 38]


In [17]:
# Test all 15 questions

import json
import os

all_answers = []

for item in questions:
    result = rag_answer(item["question"])

    record = {
        "question_id": item["id"],
        "question": item["question"],
        "expected_passage": item["gold_passage"],
        "expected_page": item["gold_page"],
        "generated_answer": result["answer"],
        "retrieved_pages": result["source_pages"]
    }

    all_answers.append(record)

    print("\n" + "=" * 70)
    print("Question:", item["question"])
    print("Expected page:", item["gold_page"])
    print("Retrieved pages:", result["source_pages"])
    print("Answer:", result["answer"])

# Save all generated answers
output_path = os.path.join(
    "Task2-RAG-From-Scratch",
    "data",
    "rag_answers.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_answers, f, indent=2, ensure_ascii=False)

print("\nAll 15 questions tested.")
print("Results saved:", output_path)



Question: What is the main purpose of the NIST AI Risk Management Framework?
Expected page: 9
Retrieved pages: [1, 2, 6, 8, 37]
Answer: The main purpose of the NIST AI Risk Management Framework is to provide guidance and tools for organizations to identify, assess, and manage risks associated with artificial intelligence (AI) technologies.

Question: How does the AI RMF define risk?
Expected page: 9
Retrieved pages: [1, 9, 15, 47]
Answer: The AI RMF defines risk as "the composite measure of an event's probability of occurring and the magnitude or degree of the consequences of the corresponding event."

Question: What does risk tolerance mean in the context of AI risk management?
Expected page: 12
Retrieved pages: [9, 12, 13]
Answer: risk tolerance means the organization's or AI actor's (see Appendix A) readiness to bear the risk in order to achieve its objectives.

Question: What should organizations do when an AI system presents unacceptable negative risk levels?
Expected page: 13
Re

In [18]:
# Improve answer generation

def rag_answer(question):
    retrieved = retrieve_top_k(
        question,
        small_chunks,
        small_embeddings,
        k=5
    )

    context = ""

    for i, result in enumerate(retrieved, start=1):
        context += (
            f"\n[PDF Page {result['page_number']}]\n"
            f"{result['text']}\n"
        )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise document question-answering system. "
                "Answer only using the supplied document passages. "
                "Do not use outside knowledge. "
                "Do not invent procedures, steps, examples, or details. "
                "If the passages do not contain the answer, clearly say "
                "that the retrieved passages are insufficient. "
                "Keep answers concise and factual. "
                "Include the PDF page number supporting your answer."
            )
        },
        {
            "role": "user",
            "content": (
                f"DOCUMENT PASSAGES:\n{context}\n\n"
                f"QUESTION: {question}\n\n"
                "Give a direct answer in 2-4 sentences. "
                "Use only facts explicitly supported by the passages."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    source_pages = sorted(set(
        result["page_number"] for result in retrieved
    ))

    return {
        "question": question,
        "answer": answer,
        "source_pages": source_pages,
        "retrieved_passages": retrieved
    }


In [19]:

test_ids = [4, 8, 10, 12]

for question_id in test_ids:
    item = next(q for q in questions if q["id"] == question_id)

    result = rag_answer(item["question"])

    print("\n" + "=" * 65)
    print("QUESTION:", item["question"])
    print("EXPECTED PAGE:", item["gold_page"])
    print("RETRIEVED PAGES:", result["source_pages"])
    print("GENERATED ANSWER:", result["answer"])



QUESTION: What should organizations do when an AI system presents unacceptable negative risk levels?
EXPECTED PAGE: 13
RETRIEVED PAGES: [6, 9, 10, 13]
GENERATED ANSWER: When an AI system presents unacceptable negative risk levels, organizations should take several actions to mitigate these risks and ensure responsible AI use. First, they should conduct thorough risk assessments to identify and prioritize potential risks associated with the AI system's operations. Next, they should establish clear guidelines and protocols for handling and mitigating these risks. Additionally, they should engage in regular training and education programs for employees and stakeholders to reinforce understanding of the risks and their mitigation strategies. Finally, they should implement robust monitoring and auditing mechanisms to continuously track and evaluate the effectiveness of risk management measures. By taking these proactive steps, organizations can

QUESTION: What practical approaches can orga

In [20]:
# Improve grounding using evidence-first prompting

def rag_answer(question):
    retrieved = retrieve_top_k(
        question,
        small_chunks,
        small_embeddings,
        k=5
    )

    # Give the model clearly separated evidence passages
    context_parts = []

    for i, result in enumerate(retrieved, start=1):
        context_parts.append(
            f"EVIDENCE {i} (PDF page {result['page_number']}):\n"
            f"{result['text']}"
        )

    context = "\n\n".join(context_parts)

    messages = [
        {
            "role": "system",
            "content": (
                "You answer questions using only the supplied evidence. "
                "Treat the evidence as the only source of truth. "
                "Do not add general knowledge or make assumptions. "
                "Do not invent lists, procedures, recommendations, "
                "or explanations. "
                "Preserve grouped terms exactly as written in the source. "
                "If evidence is insufficient, say so. "
                "Answer in one or two sentences and cite the PDF page."
            )
        },
        {
            "role": "user",
            "content": (
                f"{context}\n\n"
                f"QUESTION: {question}\n\n"
                "First identify the evidence that directly answers "
                "the question. Then provide only the supported answer."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return {
        "question": question,
        "answer": answer,
        "source_pages": sorted(set(
            r["page_number"] for r in retrieved
        ))
    }


In [21]:

for question_id in [4, 10, 12]:
    item = next(q for q in questions if q["id"] == question_id)
    result = rag_answer(item["question"])

    print("\n" + "=" * 60)
    print("QUESTION:", result["question"])
    print("SOURCE PAGES:", result["source_pages"])
    print("ANSWER:", result["answer"])



QUESTION: What should organizations do when an AI system presents unacceptable negative risk levels?
SOURCE PAGES: [6, 9, 10, 13]
ANSWER: When an AI system presents unacceptable negative risk levels, organizations should take the following actions:

1. Conduct thorough risk assessments and analyses.
2. Develop and implement robust risk management frameworks.
3. Ensure transparency and accountability regarding risk exposure.
4. Regularly update and refine risk mitigation strategies.
5. Engage stakeholders in risk communication and decision-making processes.
6. Implement continuous monitoring and evaluation mechanisms.
7. Foster a culture of informed decision-making and stakeholder engagement around AI risks.

These steps ensure organizations

QUESTION: What are the three major categories of AI bias identified by NIST?
SOURCE PAGES: [6, 23, 34, 45]
ANSWER: Systematic bias, computational bias, and statistical bias

QUESTION: Why is the GOVERN function considered cross-cutting?
SOURCE PAG

In [22]:

import os
import json
import shutil

base = "Task2-RAG-From-Scratch"
data_dir = os.path.join(base, "data")
results_dir = os.path.join(base, "results")

os.makedirs(results_dir, exist_ok=True)

# Copy experiment outputs
files_to_copy = [
    "retrieval_results_200.json",
    "retrieval_results_400.json",
    "rag_answers.json"
]

for filename in files_to_copy:
    source = os.path.join(data_dir, filename)
    destination = os.path.join(results_dir, filename)

    if os.path.exists(source):
        shutil.copy2(source, destination)
        print("Copied:", filename)
    else:
        print("Missing:", filename)

# Save retrieval metrics
metrics = {
    "document": "NIST AI RMF 1.0",
    "document_pages": 47,
    "evaluation_questions": 15,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "generation_model": "Qwen/Qwen2.5-0.5B-Instruct",
    "retrieval_method": "NumPy cosine similarity",
    "top_k": 5,
    "chunk_size_200": {
        "chunks": 114,
        "recall_at_5": 0.9333,
        "successful_questions": 14
    },
    "chunk_size_400": {
        "chunks": 66,
        "recall_at_5": 0.80,
        "successful_questions": 12
    }
}

with open(os.path.join(results_dir, "metrics_summary.json"), "w") as f:
    json.dump(metrics, f, indent=4)

print("\nResults folder prepared!")


Copied: retrieval_results_200.json
Copied: retrieval_results_400.json
Copied: rag_answers.json

Results folder prepared!


In [23]:

import os
import shutil
from google.colab import files

base_dir = "Task2-RAG-From-Scratch"

# Create ZIP for data folder
shutil.make_archive(
    "/content/Task2_data",
    "zip",
    root_dir=base_dir,
    base_dir="data"
)

# Create ZIP for results folder
shutil.make_archive(
    "/content/Task2_results",
    "zip",
    root_dir=base_dir,
    base_dir="results"
)

print("Both ZIP files created successfully!")


Both ZIP files created successfully!


In [24]:

from google.colab import files

files.download("/content/Task2_data.zip")
files.download("/content/Task2_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>